# Sprint 2 Colab Pipeline

This notebook is the primary Google Colab runner for heavy Sprint 2 execution. It is intentionally orchestration-only: raw-data staging, direct calls into the existing repository scripts, archive packaging, and optional download or private Google Drive copy.

It must not become the home of processing logic. The source of truth remains the Sprint 2 CLI scripts under `scripts/`.


## 1. Environment Setup

This section prepares the fresh Colab runtime. It only installs operational prerequisites needed to run the repository and create `.tar.zst` archives.


In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

print(f"Python: {sys.version.split()[0]}")
print(f"Working directory before clone: {Path.cwd()}")

if shutil.which("zstd") is None:
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "zstd"], check=True)
else:
    print("zstd is already available.")


## 2. Repository Setup

Edit the repository reference if you need a different branch, tag, or commit. The defaults target the public repository and the `master` branch.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/0xmillennium/text-to-sign-production.git"
REPO_REF = "master"
WORKTREE_ROOT = Path("/content/text-to-sign-production")

if WORKTREE_ROOT.exists():
    shutil.rmtree(WORKTREE_ROOT)

subprocess.run(["git", "clone", REPO_URL, str(WORKTREE_ROOT)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=WORKTREE_ROOT, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], cwd=WORKTREE_ROOT, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], cwd=WORKTREE_ROOT, check=True)
os.chdir(WORKTREE_ROOT)

current_revision = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=WORKTREE_ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Repository ready at {WORKTREE_ROOT}")
print(f"Checked out revision: {current_revision}")


## 3. Raw Data Fetch And Setup

Choose one raw-input mode:

- `public_urls`: download translation TSVs plus split archives from public How2Sign source URLs that you provide in this notebook.
- `mounted_paths`: copy already available raw inputs from user-controlled Colab or Google Drive paths.

This section resets the canonical raw staging directories inside the Colab clone before copying data into the Sprint 2 layout:

- `data/raw/how2sign/translations/`
- `data/raw/how2sign/bfh_keypoints/`


In [ ]:
from pathlib import Path

PROJECT_ROOT = WORKTREE_ROOT
RAW_ROOT = PROJECT_ROOT / "data/raw/how2sign"
TRANSLATIONS_DIR = RAW_ROOT / "translations"
BFH_KEYPOINTS_ROOT = RAW_ROOT / "bfh_keypoints"
DOWNLOAD_ROOT = Path("/content/how2sign_downloads")

RAW_INPUT_MODE = "mounted_paths"  # Change to "public_urls" when downloading from public sources.

TRANSLATION_URLS = {
    "train": "",
    "val": "",
    "test": "",
}
KEYPOINT_ARCHIVE_URLS = {
    "train": "",
    "val": "",
    "test": "",
}

MOUNTED_TRANSLATION_FILES = {
    "train": "",
    "val": "",
    "test": "",
}
MOUNTED_KEYPOINT_SPLIT_DIRS = {
    "train": "",
    "val": "",
    "test": "",
}

SPLIT_DIR_NAMES = {
    "train": "train_2D_keypoints",
    "val": "val_2D_keypoints",
    "test": "test_2D_keypoints",
}
TRANSLATION_FILE_NAMES = {
    "train": "how2sign_realigned_train.csv",
    "val": "how2sign_realigned_val.csv",
    "test": "how2sign_realigned_test.csv",
}


In [ ]:
import shutil
import subprocess
import urllib.request
from pathlib import Path
from urllib.parse import urlparse

import gdown


def _reset_directory(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def _require_values(mapping: dict[str, str], *, label: str) -> None:
    missing = [key for key, value in mapping.items() if not value]
    if missing:
        raise ValueError(f"{label} is missing values for: {', '.join(missing)}")


def _is_google_drive_url(url: str) -> bool:
    host = urlparse(url).netloc.lower()
    return host in {"drive.google.com", "docs.google.com", "drive.usercontent.google.com"}


def _download_file(url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {url} -> {destination}")
    if _is_google_drive_url(url):
        gdown.download(url=url, output=str(destination), quiet=False, fuzzy=True)
    else:
        urllib.request.urlretrieve(url, destination)


def _extract_archive(archive_path: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    if archive_path.name.endswith(".zip"):
        shutil.unpack_archive(str(archive_path), str(destination))
        return
    if archive_path.name.endswith(".tar.zst"):
        subprocess.run(["tar", "--zstd", "-xf", str(archive_path), "-C", str(destination)], check=True)
        return
    if archive_path.name.endswith((".tar", ".tar.gz", ".tgz", ".tar.xz")):
        shutil.unpack_archive(str(archive_path), str(destination))
        return
    raise ValueError(f"Unsupported archive format: {archive_path.name}")


def _find_split_dir(search_root: Path, expected_dir_name: str) -> Path:
    candidates = sorted(
        path
        for path in search_root.rglob(expected_dir_name)
        if (path / "openpose_output" / "json").exists()
    )
    if len(candidates) != 1:
        raise RuntimeError(
            f"Could not uniquely locate {expected_dir_name} under {search_root}. Found: {candidates}"
        )
    return candidates[0]


def _copy_tree(source: Path, destination: Path) -> None:
    shutil.copytree(source, destination, dirs_exist_ok=True)


_reset_directory(TRANSLATIONS_DIR)
_reset_directory(BFH_KEYPOINTS_ROOT)
DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)

if RAW_INPUT_MODE == "public_urls":
    _require_values(TRANSLATION_URLS, label="TRANSLATION_URLS")
    _require_values(KEYPOINT_ARCHIVE_URLS, label="KEYPOINT_ARCHIVE_URLS")

    for split, url in TRANSLATION_URLS.items():
        destination = TRANSLATIONS_DIR / TRANSLATION_FILE_NAMES[split]
        _download_file(url, destination)

    for split, url in KEYPOINT_ARCHIVE_URLS.items():
        archive_name = Path(urlparse(url).path).name or f"{split}_keypoints_archive.tar"
        archive_path = DOWNLOAD_ROOT / archive_name
        extract_root = DOWNLOAD_ROOT / f"{split}_extract"
        if extract_root.exists():
            shutil.rmtree(extract_root)
        _download_file(url, archive_path)
        _extract_archive(archive_path, extract_root)
        split_dir = _find_split_dir(extract_root, SPLIT_DIR_NAMES[split])
        _copy_tree(split_dir, BFH_KEYPOINTS_ROOT / SPLIT_DIR_NAMES[split])
elif RAW_INPUT_MODE == "mounted_paths":
    _require_values(MOUNTED_TRANSLATION_FILES, label="MOUNTED_TRANSLATION_FILES")
    _require_values(MOUNTED_KEYPOINT_SPLIT_DIRS, label="MOUNTED_KEYPOINT_SPLIT_DIRS")

    for split, source_path in MOUNTED_TRANSLATION_FILES.items():
        source = Path(source_path)
        destination = TRANSLATIONS_DIR / TRANSLATION_FILE_NAMES[split]
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

    for split, source_path in MOUNTED_KEYPOINT_SPLIT_DIRS.items():
        source = Path(source_path)
        destination = BFH_KEYPOINTS_ROOT / SPLIT_DIR_NAMES[split]
        _copy_tree(source, destination)
else:
    raise ValueError("RAW_INPUT_MODE must be either 'public_urls' or 'mounted_paths'.")

for split in ("train", "val", "test"):
    translation_path = TRANSLATIONS_DIR / TRANSLATION_FILE_NAMES[split]
    keypoints_dir = BFH_KEYPOINTS_ROOT / SPLIT_DIR_NAMES[split] / "openpose_output" / "json"
    if not translation_path.exists():
        raise FileNotFoundError(f"Missing translation file: {translation_path}")
    if not keypoints_dir.exists():
        raise FileNotFoundError(f"Missing keypoint directory: {keypoints_dir}")
    print(f"Ready: {translation_path}")
    print(f"Ready: {keypoints_dir}")

print(f"Canonical raw layout staged under {RAW_ROOT}")


## 4. Sprint 2 Script Execution

Colab heavy runs call the existing scripts directly instead of forcing `dvc repro`. DVC remains the repository standard for reproducibility, but direct script execution avoids unnecessary hashing overhead over very large raw trees in the Colab runtime.


In [ ]:
import shlex
import subprocess
import sys

commands = [
    [sys.executable, "scripts/prepare_raw.py"],
    [sys.executable, "scripts/normalize_keypoints.py"],
    [sys.executable, "scripts/filter_samples.py", "--config", "configs/data/filter-v1.yaml"],
    [sys.executable, "scripts/export_training_manifest.py"],
]

for command in commands:
    print(f"$ {shlex.join(command)}")
    subprocess.run(command, cwd=WORKTREE_ROOT, check=True)

print("Sprint 2 pipeline completed.")


## 5. Output Packaging

This section loads optional storage settings, then creates explicit `.tar.zst` archives for manifests, reports, and per-split processed samples.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import subprocess
import sys
import yaml

storage_config_path = WORKTREE_ROOT / "configs/storage.local.yaml"
storage_example_path = WORKTREE_ROOT / "configs/storage.example.yaml"

if storage_config_path.exists():
    storage_config = yaml.safe_load(storage_config_path.read_text(encoding="utf-8"))
    print("Loaded local storage config override.")
else:
    storage_config = yaml.safe_load(storage_example_path.read_text(encoding="utf-8"))
    print("Loaded example storage config.")

if storage_config.get("run_label") == "sprint2-colab-example-run":
    storage_config["run_label"] = datetime.now(timezone.utc).strftime("sprint2-colab-%Y%m%dT%H%M%SZ")

subprocess.run([sys.executable, "scripts/package_sprint2_outputs.py"], cwd=WORKTREE_ROOT, check=True)

archive_root = WORKTREE_ROOT / "data/archives"
archive_paths = sorted(archive_root.glob("*.tar.zst"))
for archive_path in archive_paths:
    size_mib = archive_path.stat().st_size / (1024 * 1024)
    print(f"{archive_path.name}: {size_mib:.2f} MiB")

print(
    {
        "provider": storage_config.get("provider"),
        "run_label": storage_config.get("run_label"),
        "local_download": storage_config.get("local_download"),
        "upload_to_storage": storage_config.get("upload_to_storage"),
    }
)


## 6. Download Or Private Storage Copy

The repository never commits these large outputs to GitHub. Use browser downloads for smaller archives when practical, or mount Google Drive and copy archives to a private/shared location that is not published through GitHub Pages.


In [ ]:
import shutil
from pathlib import Path

DOWNLOAD_ARCHIVE_NAMES = ["sprint2_manifests_reports.tar.zst"]
UPLOAD_ARCHIVE_NAMES = [archive_path.name for archive_path in archive_paths]

if storage_config.get("local_download", False):
    try:
        from google.colab import files
    except ModuleNotFoundError:
        print("google.colab.files is unavailable outside Colab.")
    else:
        for archive_name in DOWNLOAD_ARCHIVE_NAMES:
            archive_path = archive_root / archive_name
            if not archive_path.exists():
                raise FileNotFoundError(f"Archive not found for download: {archive_path}")
            print(f"Downloading {archive_name}")
            files.download(str(archive_path))
else:
    print("local_download is disabled in storage config.")

if storage_config.get("upload_to_storage", False):
    drive_config = storage_config.get("google_drive", {})
    mount_path = Path(drive_config.get("mount_path", "/content/drive"))
    relative_root = Path(drive_config.get("relative_root", "MyDrive/text-to-sign-production/artifacts"))
    artifact_root = Path(storage_config.get("artifact_root", "text-to-sign-production"))
    run_label = storage_config["run_label"]

    try:
        from google.colab import drive
    except ModuleNotFoundError:
        print("google.colab.drive is unavailable outside Colab.")
    else:
        drive.mount(str(mount_path), force_remount=False)
        target_root = mount_path / relative_root / artifact_root / run_label
        target_root.mkdir(parents=True, exist_ok=True)
        for archive_name in UPLOAD_ARCHIVE_NAMES:
            archive_path = archive_root / archive_name
            destination = target_root / archive_name
            shutil.copy2(archive_path, destination)
            print(f"Copied {archive_name} -> {destination}")
else:
    print("upload_to_storage is disabled in storage config.")
